# 01. Quantile Regime Detection

Detect transparent volatility regimes from rolling realized volatility quantiles. Outputs from this notebook are consumed by motif notebooks and should not be recomputed inside later stages unless saved files are missing.


## Execution Mode

`EXECUTION_MODE = "auto"` resolves to local mode on Windows and HPC mode on Linux/HPC markers such as SLURM or CUDA environment variables. Manual override is supported.


In [ ]:
EXECUTION_MODE = "auto"  # allowed: "auto", "local", "hpc"
STAGE_NAME = "01_quantile_regime_detection"

from pathlib import Path
import sys
import warnings
warnings.simplefilter("default")

def find_workflow_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "hpc_config.py").exists():
            return candidate
    fallback = Path(r"C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\HPC workflow\HPC_Regime_and_motif_discovery")
    if (fallback / "src" / "hpc_config.py").exists():
        return fallback
    raise RuntimeError("Could not locate workflow root.")

WORKFLOW_ROOT = find_workflow_root()
SRC_DIR = WORKFLOW_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from hpc_config import build_workflow_paths, detect_execution_mode, environment_info, find_project_root, load_workflow_config, output_suffix, save_config_snapshot
from io_utils import ensure_workflow_dirs, setup_stage_logger

PROJECT_ROOT = find_project_root(WORKFLOW_ROOT)
MODE = detect_execution_mode(EXECUTION_MODE, PROJECT_ROOT)
CONFIG = load_workflow_config(MODE, WORKFLOW_ROOT)
PATHS = build_workflow_paths(WORKFLOW_ROOT, PROJECT_ROOT)
ensure_workflow_dirs(PATHS)
SUFFIX = output_suffix(CONFIG)
LOGGER = setup_stage_logger(PATHS.logs, STAGE_NAME, SUFFIX)
SNAPSHOT_PATH = save_config_snapshot(CONFIG, PATHS, STAGE_NAME, MODE)

env = environment_info(PATHS)
print("Stage:", STAGE_NAME)
print("Execution mode:", MODE)
print("Project root:", PATHS.project_root)
print("Workflow root:", PATHS.workflow_root)
print("Input feature root:", PATHS.data_root)
print("Config snapshot:", SNAPSHOT_PATH)
print({"python": env["python_version"].split()[0], "platform": env["platform"], "hostname": env["hostname"], "cpu_count": env["cpu_count"], "memory_available_gb": env["memory_available_gb"], "gpu": env["gpu"]})


## Discover Feature Files

Feature parquet files are auto-discovered under `final_dataset/features` and filtered by the active run config.


In [ ]:
from data_loading import discover_feature_files
from io_utils import save_table

inventory = discover_feature_files(PATHS.project_root, CONFIG)
if inventory.empty:
    raise FileNotFoundError(f"No feature parquet files discovered under {PATHS.data_root}")

inventory_for_save = inventory.copy()
inventory_for_save["path"] = inventory_for_save["path"].astype(str)
save_table(inventory_for_save, PATHS.tables / f"input_feature_inventory{SUFFIX}.csv", PATHS.tables / f"input_feature_inventory{SUFFIX}.parquet")
print("Discovered input files:")
print(inventory_for_save[["asset", "frequency", "market", "path"]].to_string(index=False))


## Run Quantile Regimes

For each asset/frequency, compute or validate returns and volatility, assign quantile regimes, save aligned labels, summaries, transition matrices, figures, logs, and failure tables.


In [ ]:
import pandas as pd
from data_loading import apply_mode_limits, describe_feature_frame, load_feature_file
from feature_selection import choose_volatility_column, ensure_core_features
from io_utils import failure_record, safe_name, save_failure_log, save_table
from plotting_utils import plot_price_by_regime, plot_regime_distribution, plot_transition_heatmap, plot_volatility_by_regime
from quantile_regime import run_quantile_regime_detection

all_labels, all_summaries, all_transitions = [], [], []
dataset_summaries, failures = [], []
figure_count = 0
max_figures = int(CONFIG.get("plotting", {}).get("max_figures_per_notebook", 40))
max_points = int(CONFIG.get("plotting", {}).get("max_points_per_figure", 5000))

for meta in inventory.to_dict("records"):
    asset, frequency = meta["asset"], meta["frequency"]
    try:
        LOGGER.info("Quantile regimes: %s %s", asset, frequency)
        raw = load_feature_file(meta["path"])
        df = apply_mode_limits(raw, CONFIG)
        if df.empty:
            raise ValueError("No rows remain after execution-mode limits.")
        dataset_summaries.append(describe_feature_frame(meta, df))
        print(f"\n{asset} {frequency}: rows={len(df):,}, columns={df.shape[1]}, date_range={df['timestamp'].min()} -> {df['timestamp'].max()}")
        print("Columns:", ", ".join(map(str, df.columns)))

        labels, summary, transitions = run_quantile_regime_detection(df, asset, frequency, CONFIG)
        all_labels.append(labels); all_summaries.append(summary); all_transitions.append(transitions)
        stem = f"{safe_name(asset)}_{safe_name(frequency)}_quantile_regimes{SUFFIX}"
        save_table(labels, PATHS.regimes_quantile / f"{stem}.csv", PATHS.regimes_quantile / f"{stem}.parquet")

        prepared = ensure_core_features(df, rolling_window=int(CONFIG.get("quantile", {}).get("default_rolling_window", 60)))
        vol_col = choose_volatility_column(prepared, CONFIG.get("quantile", {}).get("volatility_columns"))
        for regime_method in labels["regime_method"].drop_duplicates():
            method_labels = labels[labels["regime_method"] == regime_method][["timestamp", "regime_label"]]
            plot_df = prepared.merge(method_labels, on="timestamp", how="left")
            method_summary = summary[summary["regime_method"] == regime_method]
            method_transitions = transitions[transitions["regime_method"] == regime_method]
            prefix = f"01_{safe_name(asset)}_{safe_name(frequency)}_{safe_name(regime_method)}{SUFFIX}"
            if figure_count < max_figures:
                plot_price_by_regime(plot_df, "regime_label", f"{asset} {frequency} close by {regime_method}", PATHS.figures / f"{prefix}_close_by_regime.png", max_points=max_points); figure_count += 1
            if vol_col and figure_count < max_figures:
                plot_volatility_by_regime(plot_df, vol_col, "regime_label", f"{asset} {frequency} volatility by {regime_method}", PATHS.figures / f"{prefix}_vol_by_regime.png", max_points=max_points); figure_count += 1
            if figure_count < max_figures:
                plot_regime_distribution(method_summary, f"{asset} {frequency} regime distribution", PATHS.figures / f"{prefix}_distribution.png"); figure_count += 1
            if figure_count < max_figures:
                plot_transition_heatmap(method_transitions, f"{asset} {frequency} transition matrix", PATHS.figures / f"{prefix}_transition_heatmap.png"); figure_count += 1
    except Exception as exc:
        LOGGER.exception("Quantile regime failure for %s %s", asset, frequency)
        failures.append(failure_record(STAGE_NAME, exc, asset, frequency, {"path": str(meta["path"])}))

quantile_labels = pd.concat(all_labels, ignore_index=True) if all_labels else pd.DataFrame()
quantile_summary = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
quantile_transitions = pd.concat(all_transitions, ignore_index=True) if all_transitions else pd.DataFrame()
dataset_summary = pd.DataFrame(dataset_summaries)

save_table(quantile_labels, PATHS.regimes_quantile / f"quantile_regime_labels{SUFFIX}.csv", PATHS.regimes_quantile / f"quantile_regime_labels{SUFFIX}.parquet")
save_table(quantile_summary, PATHS.regimes_quantile / f"quantile_regime_summary{SUFFIX}.csv", PATHS.regimes_quantile / f"quantile_regime_summary{SUFFIX}.parquet")
save_table(quantile_transitions, PATHS.regimes_quantile / f"quantile_transition_matrix{SUFFIX}.csv", PATHS.regimes_quantile / f"quantile_transition_matrix{SUFFIX}.parquet")
save_table(dataset_summary, PATHS.tables / f"01_quantile_dataset_summary{SUFFIX}.csv", PATHS.tables / f"01_quantile_dataset_summary{SUFFIX}.parquet")
failure_df = save_failure_log(failures, PATHS.logs / f"01_quantile_failures{SUFFIX}.csv")

print("\nSaved quantile labels:", len(quantile_labels))
print("Saved quantile summary rows:", len(quantile_summary))
print("Saved transition rows:", len(quantile_transitions))
print("Failures:", len(failure_df))
quantile_summary.head(20)
